In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q ultralytics

In [4]:
from ultralytics import YOLO
from tensorflow.keras.models import load_model

YOLO_PATH = "/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt"
LSTM_PATH = "/content/drive/MyDrive/GP/LstmModels/best_lstm.keras"

yolo_model = YOLO(YOLO_PATH)
lstm_model = load_model(LSTM_PATH)

In [5]:
import os

frames_root = "/content/drive/MyDrive/GP2/fps30_all_frames/test/Alert"

video_paths = [
    os.path.join(frames_root, d)
    for d in os.listdir(frames_root)
    if os.path.isdir(os.path.join(frames_root, d))
]

print(f"Found {len(video_paths)} frame folders")

Found 5 frame folders


In [46]:
import os

frames_root = "/content/drive/MyDrive/GP2/fps30_all_frames/test/Drowsy"

video_paths = [
    os.path.join(frames_root, d)
    for d in os.listdir(frames_root)
    if os.path.isdir(os.path.join(frames_root, d))
]

print(f"Found {len(video_paths)} frame folders")

Found 5 frame folders


In [25]:
import cv2
import numpy as np
import time
import os
import glob

det_conf = 0.44
closed_thr = 0.25
open_thr = 0.25
min_blink_frames = 3

all_features = {}

feature_start = time.time()

for video_path in video_paths:

    video_name = os.path.basename(video_path)

    print(f"\nProcessing {video_name}")

    frame_files = sorted(
        glob.glob(os.path.join(video_path, "*.jpg"))
    )

    # If your frames are png:
    # frame_files = sorted(glob.glob(os.path.join(video_path, "*.png")))

    print(f"Frames found: {len(frame_files)}")

    state_seq = [0]
    closed_conf_seq = [0.0]

    blink_features = []
    blink_count = 0

    in_blink = False
    blink_start = -1

    for frame_file in frame_files:

        frame = cv2.imread(frame_file)

        if frame is None:
            continue

        results = yolo_model.predict(
            frame,
            conf=det_conf,
            verbose=False
        )

        closed_conf = 0.0
        open_conf = 0.0

        for res in results:

            if res.boxes is not None:

                for box in res.boxes:

                    conf = float(box.conf[0])
                    cls_id = int(box.cls[0])

                    cls_name = yolo_model.names[cls_id]

                    if cls_name == "closed eyes":
                        closed_conf = max(closed_conf, conf)

                    elif cls_name == "opened eyes":
                        open_conf = max(open_conf, conf)

        state = 1 if closed_conf >= closed_thr else 0

        state_seq.append(state)
        closed_conf_seq.append(closed_conf)

        current_idx = len(state_seq) - 1

        if not in_blink and state == 1:

            blink_start = current_idx
            in_blink = True

        elif in_blink and state == 0:

            blink_end = current_idx - 1
            in_blink = False

            duration = blink_end - blink_start + 1

            if duration >= min_blink_frames:

                feat = compute_blink_features(
                    blink_start,
                    blink_end,
                    closed_conf_seq,
                    blink_count
                )

                if feat is not None:

                    blink_features.append(feat)
                    blink_count += 1

    features_array = np.array(
        blink_features,
        dtype=np.float32
    )

    all_features[video_name] = features_array

    print(
        f"Added {video_name} "
        f"Shape={features_array.shape}"
    )

np.save(
    "/content/drive/MyDrive/GP/runtime/all_features.npy",
    all_features
)

feature_end = time.time()

print(
    f"\nFeature extraction time: "
    f"{feature_end - feature_start:.2f} sec"
)

print(f"Videos saved: {len(all_features)}")


Processing A022_20260513_190541_frames
Frames found: 5613
Added A022_20260513_190541_frames Shape=(142, 4)

Processing A023_20260513_195602_frames
Frames found: 9255
Added A023_20260513_195602_frames Shape=(162, 4)

Processing A024_20260513_212736_frames
Frames found: 5923
Added A024_20260513_212736_frames Shape=(172, 4)

Processing A025_20260513_233235_frames
Frames found: 6879
Added A025_20260513_233235_frames Shape=(199, 4)

Processing A026_20260514_025227_frames
Frames found: 9243
Added A026_20260514_025227_frames Shape=(177, 4)

Feature extraction time: 1424.84 sec
Videos saved: 5


In [8]:
import cv2
import numpy as np
import time
import os
import glob

det_conf = 0.44
closed_thr = 0.25
open_thr = 0.25
min_blink_frames = 3

drowsyall_features = {}
video_times = {}

feature_startd = time.time()

for video_path in video_paths:

    video_start = time.time()

    video_name = os.path.basename(video_path)

    print(f"\nProcessing {video_name}")

    frame_files = sorted(
        glob.glob(os.path.join(video_path, "*.jpg"))
    )

    # If your frames are png:
    # frame_files = sorted(
    #     glob.glob(os.path.join(video_path, "*.png"))
    # )

    print(f"Frames found: {len(frame_files)}")

    state_seq = [0]
    closed_conf_seq = [0.0]

    blink_features = []
    blink_count = 0

    in_blink = False
    blink_start = -1

    for frame_file in frame_files:

        frame = cv2.imread(frame_file)

        if frame is None:
            continue

        results = yolo_model.predict(
            frame,
            conf=det_conf,
            verbose=False
        )

        closed_conf = 0.0
        open_conf = 0.0

        for res in results:

            if res.boxes is not None:

                for box in res.boxes:

                    conf = float(box.conf[0])
                    cls_id = int(box.cls[0])

                    cls_name = yolo_model.names[cls_id]

                    if cls_name == "closed eyes":
                        closed_conf = max(closed_conf, conf)

                    elif cls_name == "opened eyes":
                        open_conf = max(open_conf, conf)

        state = 1 if closed_conf >= closed_thr else 0

        state_seq.append(state)
        closed_conf_seq.append(closed_conf)

        current_idx = len(state_seq) - 1

        if not in_blink and state == 1:

            blink_start = current_idx
            in_blink = True

        elif in_blink and state == 0:

            blink_end = current_idx - 1
            in_blink = False

            duration = blink_end - blink_start + 1

            if duration >= min_blink_frames:

                feat = compute_blink_features(
                    blink_start,
                    blink_end,
                    closed_conf_seq,
                    blink_count
                )

                if feat is not None:

                    blink_features.append(feat)
                    blink_count += 1

    # Handle blink that continues until the last frame
    if in_blink:

        blink_end = len(state_seq) - 1
        duration = blink_end - blink_start + 1

        if duration >= min_blink_frames:

            feat = compute_blink_features(
                blink_start,
                blink_end,
                closed_conf_seq,
                blink_count
            )

            if feat is not None:

                blink_features.append(feat)
                blink_count += 1

    features_array = np.array(
        blink_features,
        dtype=np.float32
    )

    drowsyall_features[video_name] = features_array

    video_time = time.time() - video_start
    video_times[video_name] = video_time

    print(
        f"Added {video_name} | "
        f"Shape={features_array.shape} | "
        f"Time={video_time:.2f} sec"
    )

# Save features
np.save(
    "/content/drive/MyDrive/GP/runtime/alertall_features.npy",
    all_features
)

# Save runtimes
np.save(
    "/content/drive/MyDrive/GP/runtime/alert_video_times.npy",
    video_times
)

dfeature_end = time.time()

print(
    f"\nTotal feature extraction time: "
    f"{dfeature_end - feature_startd:.2f} sec"
)

print(f"Videos processed: {len(all_features)}")

print("\nPer-video runtimes:")
for video_name, runtime in video_times.items():
    print(f"{video_name}: {runtime:.2f} sec")


Processing A022_20260513_190541_frames
Frames found: 5613
Added A022_20260513_190541_frames | Shape=(142, 4) | Time=130.52 sec

Processing A023_20260513_195602_frames
Frames found: 9255
Added A023_20260513_195602_frames | Shape=(162, 4) | Time=297.34 sec

Processing A024_20260513_212736_frames
Frames found: 5923
Added A024_20260513_212736_frames | Shape=(172, 4) | Time=184.47 sec

Processing A025_20260513_233235_frames
Frames found: 6879
Added A025_20260513_233235_frames | Shape=(199, 4) | Time=215.24 sec

Processing A026_20260514_025227_frames
Frames found: 9243
Added A026_20260514_025227_frames | Shape=(177, 4) | Time=334.19 sec


NameError: name 'all_features' is not defined

In [26]:
import time
import pandas as pd
WINDOW_SIZE = 20   # or whatever value you trained with
STRIDE = 2         # or your training stride
SEQUENCE_LENGTH = 5
window_start = time.time()

windows_data = []

for video_name, blink_features in all_features.items():

    window_idx = 0

    for start in range(
        0,
        len(blink_features) - WINDOW_SIZE + 1,
        STRIDE
    ):

        window = blink_features[start:start + WINDOW_SIZE]

        windows_data.append({
            "video_id": video_name,
            "window_idx": window_idx,
            "data": window.reshape(-1).astype(np.float32)
        })

        window_idx += 1

windows_df = pd.DataFrame(windows_data)

window_end = time.time()

print(f"Number of windows: {len(windows_df)}")
print(f"Window generation time: {window_end - window_start:.4f} sec")

Number of windows: 380
Window generation time: 0.0083 sec


In [9]:
import time
import pandas as pd

WINDOW_SIZE = 20
STRIDE = 2
SEQUENCE_LENGTH = 5

window_startd = time.time()

windows_data = []
window_times = {}

for video_name, blink_features in drowsyall_features.items():

    video_window_start = time.time()

    window_idx = 0

    for start in range(
        0,
        len(blink_features) - WINDOW_SIZE + 1,
        STRIDE
    ):

        window = blink_features[start:start + WINDOW_SIZE]

        windows_data.append({
            "video_id": video_name,
            "window_idx": window_idx,
            "data": window.reshape(-1).astype(np.float32)
        })

        window_idx += 1

    video_window_time = time.time() - video_window_start
    window_times[video_name] = video_window_time

windows_dfdrowsy = pd.DataFrame(windows_data)

dwindow_end = time.time()

print(f"Number of windows: {len(windows_dfdrowsy)}")
print(f"Total window generation time: {dwindow_end - window_startd:.4f} sec")

print("\nPer-video window generation times:")
for video_name, runtime in window_times.items():
    print(f"{video_name}: {runtime:.6f} sec")

Number of windows: 380
Total window generation time: 0.0032 sec

Per-video window generation times:
A022_20260513_190541_frames: 0.000153 sec
A023_20260513_195602_frames: 0.000094 sec
A024_20260513_212736_frames: 0.000099 sec
A025_20260513_233235_frames: 0.000119 sec
A026_20260514_025227_frames: 0.000096 sec


In [27]:
sequence_start = time.time()

sequences = []
video_ids = []

for video_id, g in windows_df.groupby("video_id"):

    g = g.sort_values("window_idx").reset_index(drop=True)

    if len(g) < SEQUENCE_LENGTH:
        continue

    for i in range(len(g) - SEQUENCE_LENGTH + 1):

        seq = np.array(
            g.iloc[i:i + SEQUENCE_LENGTH]["data"].tolist(),
            dtype=np.float32
        )

        sequences.append(seq)
        video_ids.append(video_id)

X = np.array(sequences, dtype=np.float32)

sequence_end = time.time()

print(f"Number of sequences: {len(X)}")
print(f"Sequence shape: {X.shape}")
print(f"Sequence generation time: {sequence_end - sequence_start:.4f} sec")

Number of sequences: 360
Sequence shape: (360, 5, 80)
Sequence generation time: 0.0339 sec


In [10]:
sequence_startd = time.time()

sequences = []
video_ids = []

sequence_times = {}

for video_id, g in windows_dfdrowsy.groupby("video_id"):

    video_sequence_start = time.time()

    g = g.sort_values("window_idx").reset_index(drop=True)

    if len(g) < SEQUENCE_LENGTH:
        sequence_times[video_id] = time.time() - video_sequence_start
        continue

    for i in range(len(g) - SEQUENCE_LENGTH + 1):

        seq = np.array(
            g.iloc[i:i + SEQUENCE_LENGTH]["data"].tolist(),
            dtype=np.float32
        )

        sequences.append(seq)
        video_ids.append(video_id)

    sequence_times[video_id] = (
        time.time() - video_sequence_start
    )

Xd = np.array(sequences, dtype=np.float32)

dsequence_end = time.time()

print(f"Number of sequences: {len(Xd)}")
print(f"Sequence shape: {Xd.shape}")
print(
    f"Sequence generation time: "
    f"{dsequence_end - sequence_startd:.4f} sec"
)

print("\nPer-video sequence generation times:")
for video_name, runtime in sequence_times.items():
    print(f"{video_name}: {runtime:.6f} sec")

Number of sequences: 360
Sequence shape: (360, 5, 80)
Sequence generation time: 0.0230 sec

Per-video sequence generation times:
A022_20260513_190541_frames: 0.003451 sec
A023_20260513_195602_frames: 0.003503 sec
A024_20260513_212736_frames: 0.004771 sec
A025_20260513_233235_frames: 0.003735 sec
A026_20260514_025227_frames: 0.003707 sec


In [28]:
pred_start = time.time()

probas = lstm_model.predict(X, verbose=0).flatten()

pred_end = time.time()

print(f"LSTM inference time: {pred_end - pred_start:.4f} sec")

LSTM inference time: 0.8516 sec


In [11]:
pred_startd = time.time()

probas = lstm_model.predict(Xd, verbose=0).flatten()

dpred_end = time.time()

total_pred_time = dpred_end - pred_startd

print(f"LSTM inference time: {total_pred_time:.4f} sec")

# Count sequences per video
seq_counts = pd.Series(video_ids).value_counts().to_dict()

total_sequences = len(video_ids)

lstm_times = {}

for video_name, count in seq_counts.items():

    lstm_times[video_name] = (
        total_pred_time * count / total_sequences
    )

print("\nEstimated LSTM inference time per video:")
for video_name, runtime in lstm_times.items():
    print(f"{video_name}: {runtime:.6f} sec")

LSTM inference time: 0.7746 sec

Estimated LSTM inference time per video:
A025_20260513_233235_frames: 0.185043 sec
A026_20260514_025227_frames: 0.161375 sec
A024_20260513_212736_frames: 0.157072 sec
A023_20260513_195602_frames: 0.146313 sec
A022_20260513_190541_frames: 0.124797 sec


In [29]:
soft_start = time.time()

df = pd.DataFrame({
    "video_id": video_ids,
    "proba": probas
})

for video_id, g in df.groupby("video_id"):

    mean_proba = g["proba"].mean()

    final_pred = int(mean_proba >= 0.5)

    print("\n" + "=" * 60)
    print(f"Video: {video_id}")
    print("=" * 60)

    print(f"Number of segments: {len(g)}")
    print(f"Mean probability: {mean_proba:.4f}")
    print(f"Final prediction: {final_pred}")

    for segment_num, prob in enumerate(g["proba"], start=1):

        segment_pred = int(prob >= 0.5)

        print(
            f"Segment {segment_num:<3} | "
            f"Prob={prob:.4f} | "
            f"Pred={segment_pred}"
        )

soft_end = time.time()

print(
    f"\nSoft voting time: "
    f"{soft_end - soft_start:.6f} sec"
)


Video: A022_20260513_190541_frames
Number of segments: 58
Mean probability: 0.9699
Final prediction: 1
Segment 1   | Prob=0.9877 | Pred=1
Segment 2   | Prob=0.9881 | Pred=1
Segment 3   | Prob=0.9777 | Pred=1
Segment 4   | Prob=0.9854 | Pred=1
Segment 5   | Prob=0.9668 | Pred=1
Segment 6   | Prob=0.9860 | Pred=1
Segment 7   | Prob=0.9915 | Pred=1
Segment 8   | Prob=0.9920 | Pred=1
Segment 9   | Prob=0.9640 | Pred=1
Segment 10  | Prob=0.9662 | Pred=1
Segment 11  | Prob=0.9133 | Pred=1
Segment 12  | Prob=0.9232 | Pred=1
Segment 13  | Prob=0.9209 | Pred=1
Segment 14  | Prob=0.9876 | Pred=1
Segment 15  | Prob=0.9980 | Pred=1
Segment 16  | Prob=0.9837 | Pred=1
Segment 17  | Prob=0.9917 | Pred=1
Segment 18  | Prob=0.9231 | Pred=1
Segment 19  | Prob=0.9552 | Pred=1
Segment 20  | Prob=0.9610 | Pred=1
Segment 21  | Prob=0.9746 | Pred=1
Segment 22  | Prob=0.9976 | Pred=1
Segment 23  | Prob=0.9970 | Pred=1
Segment 24  | Prob=0.9653 | Pred=1
Segment 25  | Prob=0.9269 | Pred=1
Segment 26  | Prob=0.

In [12]:
soft_startd = time.time()

soft_times = {}

df = pd.DataFrame({
    "video_id": video_ids,
    "proba": probas
})

for video_id, g in df.groupby("video_id"):

    video_soft_start = time.time()

    mean_proba = g["proba"].mean()

    final_pred = int(mean_proba >= 0.5)

    print("\n" + "=" * 60)
    print(f"Video: {video_id}")
    print("=" * 60)

    print(f"Number of segments: {len(g)}")
    print(f"Mean probability: {mean_proba:.4f}")
    print(f"Final prediction: {final_pred}")

    for segment_num, prob in enumerate(g["proba"], start=1):

        segment_pred = int(prob >= 0.5)

        print(
            f"Segment {segment_num:<3} | "
            f"Prob={prob:.4f} | "
            f"Pred={segment_pred}"
        )

    soft_times[video_id] = (
        time.time() - video_soft_start
    )

dsoft_end = time.time()

print(
    f"\nSoft voting time: "
    f"{dsoft_end - soft_startd:.6f} sec"
)

print("\nPer-video soft voting times:")
for video_name, runtime in soft_times.items():
    print(f"{video_name}: {runtime:.6f} sec")


Video: A022_20260513_190541_frames
Number of segments: 58
Mean probability: 0.9553
Final prediction: 1
Segment 1   | Prob=0.9870 | Pred=1
Segment 2   | Prob=0.9810 | Pred=1
Segment 3   | Prob=0.9551 | Pred=1
Segment 4   | Prob=0.9789 | Pred=1
Segment 5   | Prob=0.9596 | Pred=1
Segment 6   | Prob=0.9762 | Pred=1
Segment 7   | Prob=0.9402 | Pred=1
Segment 8   | Prob=0.9305 | Pred=1
Segment 9   | Prob=0.9450 | Pred=1
Segment 10  | Prob=0.9335 | Pred=1
Segment 11  | Prob=0.8571 | Pred=1
Segment 12  | Prob=0.8986 | Pred=1
Segment 13  | Prob=0.9071 | Pred=1
Segment 14  | Prob=0.9687 | Pred=1
Segment 15  | Prob=0.9417 | Pred=1
Segment 16  | Prob=0.9542 | Pred=1
Segment 17  | Prob=0.9782 | Pred=1
Segment 18  | Prob=0.9842 | Pred=1
Segment 19  | Prob=0.9786 | Pred=1
Segment 20  | Prob=0.9619 | Pred=1
Segment 21  | Prob=0.9560 | Pred=1
Segment 22  | Prob=0.9897 | Pred=1
Segment 23  | Prob=0.9857 | Pred=1
Segment 24  | Prob=0.9460 | Pred=1
Segment 25  | Prob=0.9233 | Pred=1
Segment 26  | Prob=0.

In [33]:
print(
    f"Feature extraction : {feature_end - feature_start:.6f} sec\n"
    f"Window generation  : {window_end - window_start:.6f} sec\n"
    f"Sequence generation: {sequence_end - sequence_start:.6f} sec\n"
    f"LSTM inference     : {pred_end - pred_start:.6f} sec\n"
    f"Soft voting        : {soft_end - soft_start:.6f} sec\n"
    f"\nTotal runtime      : "
    f"{(feature_end - feature_start) + (window_end - window_start) + (sequence_end - sequence_start) + (sequence_end - sequence_start) + (soft_end - soft_start):.6f} sec"
)

Feature extraction : 1424.837968 sec
Window generation  : 0.008300 sec
Sequence generation: 0.033912 sec
LSTM inference     : 0.851593 sec
Soft voting        : 0.005672 sec

Total runtime      : 1424.919764 sec


In [43]:
print(
    f"Feature extraction : {dfeature_end - feature_startd:.6f} sec\n"
    f"Window generation  : {dwindow_end - window_startd:.6f} sec\n"
    f"Sequence generation: {dsequence_end - sequence_startd:.6f} sec\n"
    f"LSTM inference     : {dpred_end - pred_startd:.6f} sec\n"
    f"Soft voting        : {dsoft_end - soft_startd:.6f} sec\n"
    f"\nTotal runtime      : "
    f"{(dfeature_end - feature_startd) + (dwindow_end - window_startd) + (dsequence_end - sequence_startd) + (dsequence_end - sequence_startd) + (dsoft_end - soft_startd):.6f} sec"
)

Feature extraction : 1296.647100 sec
Window generation  : 0.002528 sec
Sequence generation: 0.025717 sec
LSTM inference     : 0.113278 sec
Soft voting        : 0.004597 sec

Total runtime      : 1296.705659 sec


In [55]:
import pandas as pd

all_video_names = set()

all_video_names.update(video_times.keys())
all_video_names.update(window_times.keys())
all_video_names.update(sequence_times.keys())
all_video_names.update(lstm_times.keys())
all_video_names.update(soft_times.keys())

runtime_rows = []

for video_name in sorted(all_video_names):

    feature_time = video_times.get(video_name, 0.0)
    window_time = window_times.get(video_name, 0.0)
    sequence_time = sequence_times.get(video_name, 0.0)
    lstm_time = lstm_times.get(video_name, 0.0)
    soft_time = soft_times.get(video_name, 0.0)

    total_time = (
        feature_time
        + window_time
        + sequence_time
        + lstm_time
        + soft_time
    )

    runtime_rows.append({
        "video_id": video_name,
        "feature_time": feature_time,
        "window_time": window_time,
        "sequence_time": sequence_time,
        "lstm_time": lstm_time,
        "soft_time": soft_time,
        "total_time": total_time
    })

runtime_df = pd.DataFrame(runtime_rows)

print(runtime_df)

runtime_df.to_csv(
    "/content/drive/MyDrive/GP/runtime/drowsy_runtime_breakdown.csv",
    index=False
)

print("\nSaved runtime breakdown.")

                      video_id  feature_time  window_time  sequence_time  \
0  D022_20260513_190626_frames    131.601317     0.000118       0.001817   
1  D023_20260513_195650_frames    180.463234     0.000058       0.001362   
2  D024_20260513_212814_frames    183.975574     0.000170       0.003433   
3  D025_20260513_233342_frames    168.416377     0.000091       0.002292   
4  D026_20260514_025307_frames    359.585236     0.000102       0.002351   

   lstm_time  soft_time  total_time  
0   0.012658   0.000351  131.616261  
1   0.011392   0.000239  180.476286  
2   0.034176   0.000422  184.013776  
3   0.020253   0.000275  168.439287  
4   0.022784   0.000382  359.610856  

Saved runtime breakdown.


In [13]:
import pandas as pd

all_video_names = set()

all_video_names.update(video_times.keys())
all_video_names.update(window_times.keys())
all_video_names.update(sequence_times.keys())
all_video_names.update(lstm_times.keys())
all_video_names.update(soft_times.keys())

runtime_rows = []

for video_name in sorted(all_video_names):

    feature_time = video_times.get(video_name, 0.0)
    window_time = window_times.get(video_name, 0.0)
    sequence_time = sequence_times.get(video_name, 0.0)
    lstm_time = lstm_times.get(video_name, 0.0)
    soft_time = soft_times.get(video_name, 0.0)

    total_time = (
        feature_time
        + window_time
        + sequence_time
        + lstm_time
        + soft_time
    )

    runtime_rows.append({
        "video_id": video_name,
        "feature_time": feature_time,
        "window_time": window_time,
        "sequence_time": sequence_time,
        "lstm_time": lstm_time,
        "soft_time": soft_time,
        "total_time": total_time
    })

runtime_df = pd.DataFrame(runtime_rows)

print(runtime_df)

runtime_df.to_csv(
    "/content/drive/MyDrive/GP/runtime/alert_runtime_breakdown.csv",
    index=False
)

print("\nSaved runtime breakdown.")

                      video_id  feature_time  window_time  sequence_time  \
0  A022_20260513_190541_frames    130.522925     0.000153       0.003451   
1  A023_20260513_195602_frames    297.338920     0.000094       0.003503   
2  A024_20260513_212736_frames    184.473301     0.000099       0.004771   
3  A025_20260513_233235_frames    215.239680     0.000119       0.003735   
4  A026_20260514_025227_frames    334.186161     0.000096       0.003707   

   lstm_time  soft_time  total_time  
0   0.124797   0.000540  130.651866  
1   0.146313   0.000394  297.489224  
2   0.157072   0.000441  184.635684  
3   0.185043   0.000399  215.428975  
4   0.161375   0.000407  334.351745  

Saved runtime breakdown.


In [56]:
print("\nAverage runtime:", runtime_df["total_time"].mean())
print("Min runtime:", runtime_df["total_time"].min())
print("Max runtime:", runtime_df["total_time"].max())


Average runtime: 204.83129320144653
Min runtime: 131.61626148223877
Max runtime: 359.6108555316925


In [14]:
print("\nAverage runtime:", runtime_df["total_time"].mean())
print("Min runtime:", runtime_df["total_time"].min())
print("Max runtime:", runtime_df["total_time"].max())


Average runtime: 232.51149888038634
Min runtime: 130.65186588631735
Max runtime: 334.3517454167207


In [44]:
print(f"{(dfeature_end - feature_startd) + (dwindow_end - window_startd) + (dsequence_end - sequence_startd) + (dsequence_end - sequence_startd) + (dsoft_end - soft_startd)+(feature_end - feature_start) + (window_end - window_start) + (sequence_end - sequence_start) + (sequence_end - sequence_start) + (soft_end - soft_start):.6f} sec" )

2721.625423 sec


In [15]:
seconds = 204.831293201446532721

minutes = seconds / 60
print(f"{minutes:.2f} min")

3.41 min


In [16]:
seconds = 232.51149888038634

minutes = seconds / 60
print(f"{minutes:.2f} min")

3.88 min


In [7]:
def compute_blink_features(si, ei, closed_conf_seq, blink_idx):
    duration = ei - si + 1

    seg = closed_conf_seq[si:ei + 1]

    if len(seg) == 0:
        return None

    bi = si + int(np.argmax(seg))

    baseline = min(
        closed_conf_seq[si],
        closed_conf_seq[ei]
    )

    amplitude = closed_conf_seq[bi] - baseline

    if ei > bi:
        velocity = (closed_conf_seq[bi] - closed_conf_seq[ei]) / (ei - bi)
    else:
        velocity = 0.0

    frequency = 100 * ((blink_idx + 1) / (ei + 1))

    return [
        float(duration),
        float(amplitude),
        float(velocity),
        float(frequency)
    ]